# Homework 24: Training PicoGPT

**Audience.** Students who can read the complete PicoGPT model from Homework 23.

**Prerequisites.** Next-token targets, cross-entropy, mini-batches, Adam, train/validation splits, and autoregressive generation.

**Learning goals.** By the end, you will be able to:

- construct random context-window batches from a token stream;
- train PicoGPT with AdamW and gradient clipping;
- compare greedy, temperature, and top-k generation;
- save a checkpoint containing the model, configuration, and tokenizer;
- distinguish a fast plumbing test from a meaningful TinyStories training run.

The default run is deliberately small and offline. It verifies the complete pipeline on a CPU. A clearly gated section at the end gives the larger Colab configuration.


## Outline

1. Load the single-file implementation
2. Split stories before constructing token streams
3. Inspect a random training batch
4. Train and evaluate the fast model
5. Generate and save a checkpoint
6. Optional TinyStories run
7. Notebook checkpoints


In [1]:
# S1: Locate the course module whether the notebook runs from the repo root or this folder
from pathlib import Path
import sys

import torch

if Path("pico_gpt.py").exists():
    COURSE_DIRECTORY = Path(".")
elif Path("Homework2026/pico_gpt.py").exists():
    COURSE_DIRECTORY = Path("Homework2026")
else:
    raise FileNotFoundError("Place pico_gpt.py beside this notebook.")

sys.path.insert(0, str(COURSE_DIRECTORY.resolve()))

from pico_gpt import (
    GPTConfig,
    PicoGPT,
    WordTokenizer,
    build_token_stream,
    count_parameters,
    estimate_loss,
    get_batch,
    load_checkpoint,
    make_demo_stories,
    save_checkpoint,
    seed_everything,
    split_stories,
    train_model,
)

DEVICE = torch.device("cpu")  # deterministic reference calculations
seed_everything(158)


## 1. A fast, structured corpus

The built-in corpus contains original, programmatically varied mini-stories. It includes dialogue, punctuation, animals, positive endings, and negative situations. It exists so every student can test the machinery without a network connection.

It is **not** evidence that a tiny model trained on a few hundred templated stories understands general English. The optional TinyStories run is the meaningful language-modeling experiment.


In [2]:
# S2: Split whole stories, then build the vocabulary from training stories only
stories = make_demo_stories(number_of_stories=320)
train_stories, validation_stories = split_stories(stories, validation_fraction=0.1)
tokenizer = WordTokenizer.from_texts(train_stories, max_vocab_size=512)

train_stream = build_token_stream(train_stories, tokenizer)
validation_stream = build_token_stream(validation_stories, tokenizer)

print("training stories:", len(train_stories))
print("validation stories:", len(validation_stories))
print("vocabulary size:", len(tokenizer))
print("training tokens:", len(train_stream))
print("first story:\n", train_stories[0])
print("all stories unique:", len(stories) == len(set(stories)))
print("train/validation disjoint:", set(train_stories).isdisjoint(validation_stories))
assert len(stories) == len(set(stories))
assert set(train_stories).isdisjoint(validation_stories)


training stories: 288
validation stories: 32
vocabulary size: 120
training tokens: 9893
first story:
 Rain fell over the garden. Diego felt afraid. The rabbit said, "Please come inside." Diego followed the rabbit. Soon they were warm and safe.
all stories unique: True
train/validation disjoint: True


Splitting stories first avoids placing one part of a story in training and another part in validation. The validation split still resembles training because this miniature corpus is highly structured; it is mainly a software check.


In [3]:
# S3: One reproducible random batch of context windows
fast_config = GPTConfig(
    vocab_size=len(tokenizer),
    block_size=48,
    d_model=64,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
)

batch_generator = torch.Generator().manual_seed(159)
batch_inputs, batch_targets = get_batch(
    train_stream,
    batch_size=4,
    block_size=fast_config.block_size,
    generator=batch_generator,
    device=DEVICE,
)

print("input shape:", tuple(batch_inputs.shape))
print("target shape:", tuple(batch_targets.shape))
print("first eight input tokens:", [tokenizer.itos[i] for i in batch_inputs[0, :8]])
print("first eight targets:", [tokenizer.itos[i] for i in batch_targets[0, :8]])
assert torch.equal(batch_inputs[:, 1:], batch_targets[:, :-1])


input shape: (4, 48)
target shape: (4, 48)
first eight input tokens: ['was', 'grateful', '.', '"', 'Thank', 'you', '!', '"']
first eight targets: ['grateful', '.', '"', 'Thank', 'you', '!', '"', 'Gia']


## 2. Train the fast model

The training loop is the same pattern used earlier in the course:

1. sample a mini-batch;
2. calculate cross-entropy;
3. zero old gradients;
4. backpropagate;
5. clip unusually large gradients;
6. take an AdamW step.

We report both training and validation loss. The exact decimal values can vary slightly across PyTorch versions; the important smoke test is that the loss falls substantially.


In [4]:
# S4: Train a small CPU model from scratch
FAST_STEPS = 120

seed_everything(158)
untrained_model = PicoGPT(fast_config).to(DEVICE)
initial_validation_loss = estimate_loss(
    untrained_model,
    validation_stream,
    batches=3,
    batch_size=24,
    seed=160,
)

model, history = train_model(
    fast_config,
    train_stream,
    validation_stream,
    steps=FAST_STEPS,
    batch_size=24,
    learning_rate=3e-3,
    weight_decay=0.01,
    seed=158,
    device=DEVICE,
    report_every=30,
)

# Use the same sampled windows as the initial measurement.
final_validation_loss = estimate_loss(
    model,
    validation_stream,
    batches=3,
    batch_size=24,
    seed=160,
)
print("initial validation loss:", round(initial_validation_loss, 3))
print("final validation loss:", round(final_validation_loss, 3))
assert final_validation_loss < initial_validation_loss


step    0 | train 4.797 | validation 4.795


step   30 | train 1.842 | validation 1.863


step   60 | train 1.021 | validation 1.063


step   90 | train 0.675 | validation 0.701


step  120 | train 0.579 | validation 0.604
initial validation loss: 4.795
final validation loss: 0.599


In [5]:
# S5: Inspect the loss history without printing hundreds of steps
for row in history:
    print(
        f"step {int(row['step']):3d}: "
        f"train={row['train_loss']:.3f}, validation={row['validation_loss']:.3f}"
    )


step   0: train=4.797, validation=4.795
step  30: train=1.842, validation=1.863
step  60: train=1.021, validation=1.063
step  90: train=0.675, validation=0.701
step 120: train=0.579, validation=0.604


## 3. Generation

Greedy generation always takes the largest logit. Temperature sampling draws from the full probability distribution; lower temperatures sharpen it. Top-k sampling first removes every token except the `k` most likely choices.


In [6]:
# S6: Generate from one prompt in three ways
prompt_text = "Ava found a blue kite"
prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)[:-1]
prompt = torch.tensor([prompt_ids], dtype=torch.long)

greedy_ids = model.generate(
    prompt.clone(),
    max_new_tokens=30,
    temperature=0,
    eos_id=tokenizer.eos_id,
)
sampled_ids = model.generate(
    prompt.clone(),
    max_new_tokens=30,
    temperature=0.8,
    top_k=8,
    eos_id=tokenizer.eos_id,
    seed=158,
)

untrained_ids = untrained_model.generate(
    prompt.clone(),
    max_new_tokens=30,
    temperature=0,
    eos_id=tokenizer.eos_id,
)

print("UNTRAINED:", tokenizer.decode(untrained_ids[0]))
print("GREEDY:   ", tokenizer.decode(greedy_ids[0]))
print("SAMPLED:  ", tokenizer.decode(sampled_ids[0]))


UNTRAINED: Ava found a blue kite kite kite kite kite you you you you you you gold friends friends friends friends friends friends friends friends friends friends friends friends friends friends friends friends friends friends friends
GREEDY:    Ava found a blue kite near the beach. A little turtle watched quietly. Diego shared the map. "What a good day!" Diego said. Diego danced.
SAMPLED:   Ava found a blue kite near the beach. A little goat watched quietly. Finn carried the map. "What a good day!" Diego said. Diego followed the dog watched


Save your trained model under a **student** filename. The later graded calculations use an immutable instructor checkpoint, so rerunning or modifying this training notebook cannot silently change their answers.


In [7]:
# S7: Save everything required to reproduce the trained model
checkpoint_path = COURSE_DIRECTORY / "pico_gpt_student.pt"
save_checkpoint(
    checkpoint_path,
    model,
    tokenizer,
    metadata={
        "purpose": "Student fast offline PicoGPT run",
        "training_steps": FAST_STEPS,
        "initial_validation_loss": initial_validation_loss,
        "final_validation_loss": final_validation_loss,
    },
)
print("saved:", checkpoint_path)
print("checkpoint size (KB):", round(checkpoint_path.stat().st_size / 1024, 1))


saved: pico_gpt_student.pt
checkpoint size (KB): 450.7


## 4. Optional: train the meaningful model on TinyStories

The following cell is disabled by default. In Colab, first install the dataset loader with `pip install datasets`, enable a GPU runtime, and change `RUN_TINYSTORIES` to `True`.

The intended course model uses a 4,096-token word/punctuation vocabulary, context length 128, four layers, four heads, and width 256—about 4.3 million parameters. Start with a small story limit while testing, then benchmark before assigning a longer run. Colab hardware is not guaranteed, so later interpretability assignments should use an instructor-provided canonical checkpoint.

Background: [TinyStories paper](https://arxiv.org/abs/2305.07759) and [official dataset](https://huggingface.co/datasets/roneneldan/TinyStories).


In [8]:
# S8: Optional TinyStories configuration and training path
RUN_TINYSTORIES = False

tiny_stories_config = GPTConfig(
    vocab_size=4096,
    block_size=128,
    d_model=256,
    n_heads=4,
    n_layers=4,
    dropout=0.0,
)
proposed_model = PicoGPT(tiny_stories_config)
tiny_stories_parameter_count = count_parameters(proposed_model)
del proposed_model

print("proposed TinyStories model parameters:", f"{tiny_stories_parameter_count:,}")

if RUN_TINYSTORIES:
    from itertools import islice
    from datasets import load_dataset

    if torch.cuda.is_available():
        full_device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        full_device = torch.device("mps")
    else:
        full_device = torch.device("cpu")

    streamed_train = load_dataset(
        "roneneldan/TinyStories",
        split="train",
        streaming=True,
    )
    streamed_validation = load_dataset(
        "roneneldan/TinyStories",
        split="validation",
        streaming=True,
    )
    full_train = [row["text"] for row in islice(streamed_train, 50_000)]
    full_validation = [
        row["text"] for row in islice(streamed_validation, 5_000)
    ]
    full_tokenizer = WordTokenizer.from_texts(
        full_train, max_vocab_size=4096
    )
    full_train_stream = build_token_stream(full_train, full_tokenizer)
    full_validation_stream = build_token_stream(
        full_validation, full_tokenizer
    )

    # Rebuild because the exact vocabulary size can be below 4096.
    full_config = GPTConfig(
        vocab_size=len(full_tokenizer),
        block_size=128,
        d_model=256,
        n_heads=4,
        n_layers=4,
        dropout=0.0,
    )
    full_model, full_history = train_model(
        full_config,
        full_train_stream,
        full_validation_stream,
        steps=4000,
        batch_size=32,
        learning_rate=3e-4,
        seed=158,
        device=full_device,
        report_every=400,
    )
    save_checkpoint(
        COURSE_DIRECTORY / "pico_gpt_tinystories_student.pt",
        full_model,
        full_tokenizer,
        metadata={"training_steps": 4000, "story_limit": 50_000},
    )
else:
    print("TinyStories training skipped. The offline smoke run above is complete.")


proposed TinyStories model parameters: 4,240,896
TinyStories training skipped. The offline smoke run above is complete.


## Notebook checkpoints

The stable questions should concern shapes, data flow, configuration, and fixed-checkpoint predictions—not the subjective quality of sampled prose.


In [9]:
# S9: Fixed reference-checkpoint probabilities and a checkpoint record
tiny_reference_path = COURSE_DIRECTORY / "pico_gpt_tinystories_reference.pt"
fast_reference_path = COURSE_DIRECTORY / "pico_gpt_reference.pt"
reference_path = (
    tiny_reference_path if tiny_reference_path.exists() else fast_reference_path
)
if not reference_path.exists():
    raise FileNotFoundError(
        "The immutable course reference checkpoint is missing."
    )

reference_model, reference_tokenizer, reference_metadata = load_checkpoint(
    reference_path
)
reference_prompt_ids = [reference_tokenizer.bos_id] + reference_tokenizer.encode(
    prompt_text, add_special_tokens=False
)
reference_prompt = torch.tensor([reference_prompt_ids])

with torch.no_grad():
    prompt_logits, _ = reference_model(reference_prompt)
    final_probabilities = torch.softmax(prompt_logits[0, -1], dim=-1)
    top_probabilities, top_ids = torch.topk(final_probabilities, k=5)

top_five = [
    (reference_tokenizer.itos[int(index)], round(float(probability), 4))
    for probability, index in zip(top_probabilities, top_ids)
]

checkpoint_24 = {
    "batch_input_shape": tuple(batch_inputs.shape),
    "batch_target_shape": tuple(batch_targets.shape),
    "student_parameter_count": count_parameters(model),
    "loss_decreased": final_validation_loss < initial_validation_loss,
    "reference_checkpoint": reference_path.name,
    "reference_prompt_token_count": reference_prompt.shape[1],
    "reference_top_five_next_tokens": top_five,
    "tiny_stories_parameter_count": tiny_stories_parameter_count,
}
checkpoint_24


{'batch_input_shape': (4, 48),
 'batch_target_shape': (4, 48),
 'student_parameter_count': 110848,
 'loss_decreased': True,
 'reference_checkpoint': 'pico_gpt_reference.pt',
 'reference_prompt_token_count': 6,
 'reference_top_five_next_tokens': [('near', 0.8358),
  ('at', 0.056),
  ('.', 0.0308),
  ('the', 0.022),
  ('with', 0.01)],
 'tiny_stories_parameter_count': 4240896}

## Pitfall and extension

**Pitfall.** Do not flatten the corpus and then split it at an arbitrary token. That can put adjacent pieces of the same story on opposite sides of the train/validation boundary.

**Optional extension.** Hold the trained checkpoint fixed and compare several temperatures using the same random seed. Record repetition, punctuation frequency, and output length in addition to reading the samples.
